# S6E9: OOF ensemble

複数の03_finalize OutputをCPUでcross-fit比較します。


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

required = ['numpy', 'pandas', 'scipy', 'sklearn']
missing = [name for name in required if importlib.util.find_spec(name) is None]
assert not missing, f'Missing libraries: {missing}'
INPUT_ROOT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/s6e9_ensemble_v1')
MIN_FOLD_WINS = 4
print('Inputs:', INPUT_ROOT, 'Output:', OUTPUT)


In [ ]:
ENSEMBLE_SCRIPT = Path('/kaggle/working/ensemble.py')
ENSEMBLE_SCRIPT.write_text('"""CPU-only cross-fitted ensemble over completed S6E9 Notebook outputs."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\n\nTARGET = "Will_Buy_EV"\nVARIANTS = (\n    ("chosen", "prediction", "submission.csv", None),\n    ("probability", "probability_prediction", "submission_probability.csv", "probability"),\n    ("rank", "rank_prediction", "submission_rank.csv", "rank"),\n)\n\n\ndef digest(path):\n    value = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False,\n                                    allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef binary_auc(y, prediction):\n    y = np.asarray(y)\n    prediction = np.asarray(prediction, dtype=float)\n    if y.ndim != 1 or prediction.shape != y.shape or not np.isfinite(prediction).all():\n        raise ValueError("AUC inputs must be aligned finite vectors")\n    positives = int(np.count_nonzero(y == 1))\n    negatives = int(np.count_nonzero(y == 0))\n    if positives + negatives != len(y) or not positives or not negatives:\n        raise ValueError("AUC labels must contain both binary classes")\n    order = np.argsort(prediction, kind="quicksort")\n    ordered_prediction, ordered_y = prediction[order], y[order]\n    starts = np.r_[0, 1 + np.flatnonzero(ordered_prediction[1:] != ordered_prediction[:-1])]\n    ends = np.r_[starts[1:], len(y)]\n    counts = np.add.reduceat(ordered_y, starts)\n    rank_sum = np.sum(counts * ((starts + 1 + ends) * .5))\n    return float((rank_sum - positives * (positives + 1) * .5) / (positives * negatives))\n\n\ndef _validated_frame(path, columns):\n    frame = pd.read_csv(path)\n    missing = set(columns) - set(frame)\n    if missing or frame[list(columns)].isna().any().any():\n        raise ValueError(f"Invalid {path}: missing={sorted(missing)}")\n    return frame\n\n\ndef discover_runs(input_root):\n    root = Path(input_root)\n    runs = []\n    for oof_path in sorted(root.rglob("final_oof.csv")) if root.is_dir() else []:\n        folder = oof_path.parent\n        required = [folder / name for name in ("config.json", "final_report.json", "submission.csv")]\n        if all(path.is_file() for path in required):\n            runs.append(folder)\n    if len(runs) < 2:\n        raise FileNotFoundError(\n            "Add at least two completed 03_finalize/monolithic Notebook Outputs as Input; "\n            f"found {len(runs)} valid run folder(s).")\n    return runs\n\n\ndef load_candidates(input_root):\n    runs = discover_runs(input_root)\n    base_ids = base_test_ids = test_output_ids = labels = folds = data_hashes = id_col = None\n    candidates, seen = [], set()\n    for folder in runs:\n        config = json.loads((folder / "config.json").read_text(encoding="utf-8"))\n        report = json.loads((folder / "final_report.json").read_text(encoding="utf-8"))\n        hashes = config.get("hashes")\n        if not isinstance(hashes, dict):\n            raise ValueError(f"Missing data hashes: {folder}")\n        submission = pd.read_csv(folder / "submission.csv")\n        ids = [name for name in submission if name != TARGET]\n        if len(ids) != 1:\n            raise ValueError(f"Expected one submission ID column: {folder}")\n        current_id = ids[0]\n        oof = _validated_frame(folder / "final_oof.csv", [current_id, TARGET, "fold", "prediction"])\n        if oof[current_id].duplicated().any() or submission[current_id].duplicated().any():\n            raise ValueError(f"Duplicate IDs: {folder}")\n        oof.index = oof[current_id].astype(str)\n        submission.index = submission[current_id].astype(str)\n        if base_ids is None:\n            id_col = current_id\n            base_ids = oof.index.to_numpy(copy=True)\n            base_test_ids = submission.index.to_numpy(copy=True)\n            test_output_ids = submission[current_id].to_numpy(copy=True)\n            labels = oof.loc[base_ids, TARGET].to_numpy(dtype=int)\n            folds = oof.loc[base_ids, "fold"].to_numpy(dtype=int)\n            data_hashes = hashes\n            if set(np.unique(labels)) != {0, 1} or set(np.unique(folds)) != set(range(5)):\n                raise ValueError("Expected binary labels and exactly five final OOF folds")\n        else:\n            if current_id != id_col or hashes != data_hashes:\n                raise ValueError(f"Data schema/hash mismatch: {folder}")\n            if set(oof.index) != set(base_ids) or set(submission.index) != set(base_test_ids):\n                raise ValueError(f"Train/test ID mismatch: {folder}")\n            if not np.array_equal(oof.loc[base_ids, TARGET].to_numpy(dtype=int), labels):\n                raise ValueError(f"Target mismatch: {folder}")\n            if not np.array_equal(oof.loc[base_ids, "fold"].to_numpy(dtype=int), folds):\n                raise ValueError(f"Fold mismatch: {folder}")\n        source = folder.relative_to(Path(input_root)).as_posix()\n        for variant, oof_column, submission_name, report_key in VARIANTS:\n            submission_path = folder / submission_name\n            if oof_column not in oof or not submission_path.is_file():\n                continue\n            variant_submission = pd.read_csv(submission_path)\n            if set(variant_submission) != {id_col, TARGET} or variant_submission[id_col].duplicated().any():\n                raise ValueError(f"Invalid submission schema: {submission_path}")\n            variant_submission.index = variant_submission[id_col].astype(str)\n            if set(variant_submission.index) != set(base_test_ids):\n                raise ValueError(f"Submission ID mismatch: {submission_path}")\n            expected_hash = (report.get("submission_sha256") if report_key is None else\n                report.get("cpu_submission_variants", {}).get(report_key, {}).get("submission_sha256"))\n            if expected_hash is not None and digest(submission_path) != expected_hash:\n                raise ValueError(f"Submission hash mismatch: {submission_path}")\n            train_prediction = oof.loc[base_ids, oof_column].to_numpy(dtype=float)\n            test_prediction = variant_submission.loc[base_test_ids, TARGET].to_numpy(dtype=float)\n            if (not np.isfinite(train_prediction).all() or not np.isfinite(test_prediction).all()\n                    or ((train_prediction < 0) | (train_prediction > 1)).any()\n                    or ((test_prediction < 0) | (test_prediction > 1)).any()):\n                raise ValueError(f"Invalid predictions: {folder}/{variant}")\n            identity = hashlib.sha256(train_prediction.tobytes() + test_prediction.tobytes()).hexdigest()\n            if identity in seen:\n                continue\n            seen.add(identity)\n            candidates.append(dict(name=f"{source}:{variant}", oof=train_prediction,\n                                   test=test_prediction, source=source, variant=variant))\n    if len(candidates) < 2:\n        raise ValueError("Fewer than two unique aligned prediction candidates were found")\n    return dict(id_col=id_col, train_ids=base_ids, test_ids=test_output_ids,\n                labels=labels, folds=folds, data_hashes=data_hashes, candidates=candidates)\n\n\ndef rank_matrix_by_fold(matrix, folds):\n    result = np.empty_like(matrix, dtype=float)\n    for fold in sorted(np.unique(folds)):\n        rows = folds == fold\n        for column in range(matrix.shape[1]):\n            result[rows, column] = (rankdata(matrix[rows, column], method="average") - .5) / rows.sum()\n    return result\n\n\ndef rank_matrix(matrix):\n    result = np.empty_like(matrix, dtype=float)\n    for column in range(matrix.shape[1]):\n        result[:, column] = (rankdata(matrix[:, column], method="average") - .5) / len(matrix)\n    return result\n\n\ndef best_single(y, values, rows):\n    scores = [binary_auc(y[rows], values[rows, column]) for column in range(values.shape[1])]\n    index = int(np.argmax(scores))\n    return float(scores[index]), index\n\n\ndef best_pair(y, values, rows, shortlist=6, weights=(.25, .5, .75)):\n    single_scores = np.array([binary_auc(y[rows], values[rows, column])\n                              for column in range(values.shape[1])])\n    order = np.argsort(-single_scores, kind="stable")[:min(shortlist, values.shape[1])]\n    best = dict(score=float(single_scores[order[0]]), indices=[int(order[0])], weights=[1.])\n    for position, left in enumerate(order):\n        for right in order[position + 1:]:\n            for weight in weights:\n                prediction = weight * values[rows, left] + (1 - weight) * values[rows, right]\n                score = binary_auc(y[rows], prediction)\n                if score > best["score"] + 1e-12:\n                    best = dict(score=score, indices=[int(left), int(right)],\n                                weights=[float(weight), float(1 - weight)])\n    return best\n\n\ndef refine_pair(y, values, rows, choice):\n    if len(choice["indices"]) != 2:\n        return choice\n    left, right = choice["indices"]\n    best = dict(choice)\n    for step in range(1, 20):\n        weight = step / 20\n        prediction = weight * values[rows, left] + (1 - weight) * values[rows, right]\n        score = binary_auc(y[rows], prediction)\n        if score > best["score"] + 1e-12:\n            best = dict(score=score, indices=[left, right], weights=[weight, 1 - weight])\n    return best\n\n\ndef crossfit_mode(y, values, folds, min_fold_wins=4):\n    baseline_prediction = np.empty(len(y), dtype=float)\n    blend_prediction = np.empty(len(y), dtype=float)\n    selections = []\n    for fold in sorted(np.unique(folds)):\n        training, validation = folds != fold, folds == fold\n        _, baseline = best_single(y, values, training)\n        blend = best_pair(y, values, training)\n        baseline_prediction[validation] = values[validation, baseline]\n        blend_prediction[validation] = sum(weight * values[validation, index]\n            for index, weight in zip(blend["indices"], blend["weights"]))\n        selections.append(dict(fold=int(fold), baseline=int(baseline),\n                               indices=blend["indices"], weights=blend["weights"]))\n    baseline_auc = binary_auc(y, baseline_prediction)\n    blend_auc = binary_auc(y, blend_prediction)\n    baseline_folds, blend_folds = [], []\n    for fold in sorted(np.unique(folds)):\n        rows = folds == fold\n        baseline_folds.append(binary_auc(y[rows], baseline_prediction[rows]))\n        blend_folds.append(binary_auc(y[rows], blend_prediction[rows]))\n    wins = sum(right > left + 1e-6 for left, right in zip(baseline_folds, blend_folds))\n    return dict(allowed=blend_auc > baseline_auc + 1e-6 and wins >= min_fold_wins,\n        baseline_auc=baseline_auc, blend_auc=blend_auc, delta=blend_auc - baseline_auc,\n        fold_wins=wins, required_fold_wins=min_fold_wins,\n        baseline_fold_auc=baseline_folds, blend_fold_auc=blend_folds, selections=selections)\n\n\ndef build_ensemble(bundle, output_dir, min_fold_wins=4):\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    y, folds = bundle["labels"], bundle["folds"]\n    raw_oof = np.column_stack([candidate["oof"] for candidate in bundle["candidates"]])\n    raw_test = np.column_stack([candidate["test"] for candidate in bundle["candidates"]])\n    modes = {\n        "probability": (raw_oof, raw_test),\n        "rank": (rank_matrix_by_fold(raw_oof, folds), rank_matrix(raw_test)),\n    }\n    diagnostics = {mode: crossfit_mode(y, values[0], folds, min_fold_wins)\n                   for mode, values in modes.items()}\n    names = [candidate["name"] for candidate in bundle["candidates"]]\n    for result in diagnostics.values():\n        for selection in result["selections"]:\n            selection["baseline_candidate"] = names[selection["baseline"]]\n            selection["candidates"] = [names[index] for index in selection["indices"]]\n    allowed = [mode for mode, result in diagnostics.items() if result["allowed"]]\n    chosen_mode = max(allowed, key=lambda mode: diagnostics[mode]["blend_auc"]) if allowed else None\n    report = dict(accepted=False, chosen_mode=chosen_mode,\n        candidates=[dict(name=name, auc=binary_auc(y, raw_oof[:, index]),\n            fold_auc=[binary_auc(y[folds == fold], raw_oof[folds == fold, index])\n                      for fold in sorted(np.unique(folds))])\n            for index, name in enumerate(names)],\n        data_hashes=bundle["data_hashes"], crossfit=diagnostics,\n        note="OOF model/blend selection remains selection-biased; Public LB was not used.")\n    submission_path = output / "submission_ensemble.csv"\n    submission_path.unlink(missing_ok=True)\n    if chosen_mode is not None:\n        oof_values, test_values = modes[chosen_mode]\n        rows = np.ones(len(y), dtype=bool)\n        _, global_baseline = best_single(y, oof_values, rows)\n        blend = refine_pair(y, oof_values, rows, best_pair(y, oof_values, rows))\n        global_baseline_auc = binary_auc(y, oof_values[:, global_baseline])\n        accepted = (len(blend["indices"]) == 2\n                    and blend["score"] > global_baseline_auc + 1e-6)\n        report.update(accepted=accepted, global_baseline=dict(\n            candidate=bundle["candidates"][global_baseline]["name"], auc=global_baseline_auc),\n            final_blend=dict(auc=blend["score"], indices=blend["indices"], weights=blend["weights"],\n                candidates=[bundle["candidates"][index]["name"] for index in blend["indices"]]))\n        if accepted:\n            prediction = sum(weight * test_values[:, index]\n                for index, weight in zip(blend["indices"], blend["weights"]))\n            pd.DataFrame({bundle["id_col"]: bundle["test_ids"], TARGET: prediction}).to_csv(\n                submission_path, index=False)\n            report["submission_sha256"] = digest(submission_path)\n    write_json(output / "ensemble_report.json", report)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output", default="/kaggle/working/s6e9_ensemble_v1")\n    parser.add_argument("--min-fold-wins", type=int, default=4)\n    args = parser.parse_args()\n    if not 1 <= args.min_fold_wins <= 5:\n        parser.error("min-fold-wins must be 1..5")\n    bundle = load_candidates(args.input_root)\n    report = build_ensemble(bundle, args.output, args.min_fold_wins)\n    print(json.dumps(report, indent=2, ensure_ascii=False))\n    if not report["accepted"]:\n        print("No ensemble submission was created because the cross-fit/full-OOF gates did not pass.")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')


In [ ]:
subprocess.run([sys.executable, '-u', str(ENSEMBLE_SCRIPT),
    '--input-root', str(INPUT_ROOT), '--output', str(OUTPUT),
    '--min-fold-wins', str(MIN_FOLD_WINS)], check=True)


In [ ]:
from IPython.display import FileLink, display

report = json.loads((OUTPUT / 'ensemble_report.json').read_text(encoding='utf-8'))
print(json.dumps(report, indent=2, ensure_ascii=False))
submission = OUTPUT / 'submission_ensemble.csv'
if report['accepted']:
    display(FileLink(str(submission)))
else:
    print('Cross-fit gate未通過のため、提出CSVは生成していません。')
